# Adult Income - Live Prediction (Jupyter)

This notebook is a Jupyter-friendly version of the Streamlit `else` (Live prediction) section.

In [5]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import pandas as pd

# Make local package imports work in Jupyter even if CWD is not project root
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
project_root = next((p for p in candidates if (p / "src").exists()), None)
if project_root is None:
    raise FileNotFoundError("Could not locate project root containing 'src' folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import BEST_MODEL_FILE, TARGET_LABELS
from src.data_utils import load_adult_train_test

train_df, test_df = load_adult_train_test()
if not BEST_MODEL_FILE.exists():
    raise FileNotFoundError("Model file not found. Run train_model.py first.")

model = joblib.load(BEST_MODEL_FILE)
print("Model loaded from:", BEST_MODEL_FILE)
print("Using project root:", project_root)

Model loaded from: C:\Users\User\ML-CW\models\best_model.joblib
Using project root: C:\Users\User\ML-CW


In [2]:
# Available categories from training data
choices = {
    "workclass": sorted(train_df["workclass"].dropna().unique().tolist()),
    "education": sorted(train_df["education"].dropna().unique().tolist()),
    "marital_status": sorted(train_df["marital_status"].dropna().unique().tolist()),
    "occupation": sorted(train_df["occupation"].dropna().unique().tolist()),
    "relationship": sorted(train_df["relationship"].dropna().unique().tolist()),
    "race": sorted(train_df["race"].dropna().unique().tolist()),
    "sex": sorted(train_df["sex"].dropna().unique().tolist()),
    "native_country": sorted(train_df["native_country"].dropna().unique().tolist()),
}

for k, v in choices.items():
    print(f"{k}: {v[:6]} ... (total {len(v)})")

workclass: ['Federal-gov', 'Local-gov', 'Never-worked', 'Private', 'Self-emp-inc', 'Self-emp-not-inc'] ... (total 8)
education: ['10th', '11th', '12th', '1st-4th', '5th-6th', '7th-8th'] ... (total 16)
marital_status: ['Divorced', 'Married-AF-spouse', 'Married-civ-spouse', 'Married-spouse-absent', 'Never-married', 'Separated'] ... (total 7)
occupation: ['Adm-clerical', 'Armed-Forces', 'Craft-repair', 'Exec-managerial', 'Farming-fishing', 'Handlers-cleaners'] ... (total 14)
relationship: ['Husband', 'Not-in-family', 'Other-relative', 'Own-child', 'Unmarried', 'Wife'] ... (total 6)
race: ['Amer-Indian-Eskimo', 'Asian-Pac-Islander', 'Black', 'Other', 'White'] ... (total 5)
sex: ['Female', 'Male'] ... (total 2)
native_country: ['Cambodia', 'Canada', 'China', 'Columbia', 'Cuba', 'Dominican-Republic'] ... (total 41)


In [3]:
def predict_income(input_row: dict) -> tuple[str, float, pd.DataFrame]:
    """Predict income class for one person and return label, confidence, and row dataframe."""
    row_df = pd.DataFrame([input_row])
    pred = int(model.predict(row_df)[0])
    prob = model.predict_proba(row_df)[0]
    confidence = float(prob[pred])
    label = TARGET_LABELS[pred]
    return label, confidence, row_df

sample = {
    "age": 37,
    "workclass": choices["workclass"][0],
    "fnlwgt": 189778,
    "education": choices["education"][0],
    "education_num": 10,
    "marital_status": choices["marital_status"][0],
    "occupation": choices["occupation"][0],
    "relationship": choices["relationship"][0],
    "race": choices["race"][0],
    "sex": choices["sex"][0],
    "capital_gain": 0,
    "capital_loss": 0,
    "hours_per_week": 40,
    "native_country": choices["native_country"][0],
}

label, confidence, row = predict_income(sample)
print(f"Predicted income class: {label}")
print(f"Model confidence: {confidence:.1%}")
row

Predicted income class: <=50K
Model confidence: 86.8%


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,37,Federal-gov,189778,10th,10,Divorced,Adm-clerical,Husband,Amer-Indian-Eskimo,Female,0,0,40,Cambodia


In [4]:
# Optional: keep prediction history in the notebook session
if "prediction_history" not in globals():
    prediction_history = []

hist = row.copy()
hist["prediction"] = label
hist["confidence"] = round(confidence, 4)
prediction_history.append(hist)

history_df = pd.concat(prediction_history, ignore_index=True)
history_df

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,prediction,confidence
0,37,Federal-gov,189778,10th,10,Divorced,Adm-clerical,Husband,Amer-Indian-Eskimo,Female,0,0,40,Cambodia,<=50K,0.8682
